# FirstAI + Orbit — integrated engineering foundation\nRun the cells from top to bottom. This notebook verifies the custom GPT foundation, runs the GPU training pipeline, and includes tests for Orbit's identity, task recovery, persistent memory, and tool authorization. The baseline checkpoint is still an experimental language model, not a capable assistant.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = Path('/content/FirstAI')
if REPO.is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/llakshyaa-rgb/FirstAI.git', str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import torch
print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi is not available in this runtime.')

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA runtime:', torch.version.cuda)
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GiB')
else:
    print('\nGPU CHECK NOT PASSED.')
    print('In Colab choose Runtime → Change runtime type → T4 GPU, save,')
    print('then Runtime → Disconnect and delete runtime, reconnect, and rerun from the top.')
    print('If nvidia-smi shows a GPU but PyTorch CUDA build is None, this runtime has CPU-only PyTorch.')
    print('Do not run Test 2 until CUDA available is True.')

# All training data and checkpoints use temporary Colab storage; no Drive mount or uploads are needed.
LOCAL_BASE = Path('/content/FirstAI_workspace')
BASE = LOCAL_BASE
RAW = BASE / 'data' / 'raw'
OUT = BASE / 'checkpoints'
CORPUS = str(BASE / 'data' / 'input.txt')
CHECKPOINT = str(OUT / 'firstgpt.pt')
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print('\nInitial workspace:', BASE)

## Test 1 — automated FirstAI and Orbit unit tests\nChecks tokenizer, corpus preparation, data batching, model shape, causal attention, gradients, Orbit identity, task fallback/recovery, memory persistence, tool authorization, specialist-agent routing, and inference resource budgets.

In [ ]:
compile_result = subprocess.run([
    sys.executable, '-m', 'compileall', '-q', 'firstai', 'orbit',
    'train.py', 'generate.py', 'evaluate.py', 'finetune_instructions.py', 'download_training_data.py'
], check=False)
if compile_result.returncode:
    raise RuntimeError('Python syntax compilation failed.')
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## Test 2 — training engine and checkpoint recovery
Runs a tiny temporary training job, saves latest/best checkpoints, resumes, restores scaler/RNG state, evaluates and generates. This is an engineering test, not meaningful knowledge training.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        'Test 2 needs a CUDA-enabled Colab runtime. The current PyTorch build is '
        f'{torch.__version__} (CUDA build: {torch.version.cuda}). '
        'Select T4 GPU, reconnect the runtime, and rerun the setup cell.'
    )
!python scripts/smoke_test.py --require-cuda

## Test 3 — integrated GPU pipeline with a generated test corpus
No Google Drive, uploads or external dataset are needed. This test creates a temporary synthetic text corpus inside Colab, prepares it, trains from random initialization, resumes, evaluates and generates. It verifies the engineering pipeline only; synthetic text does not teach useful real-world knowledge.

In [ ]:
# Use only temporary Colab storage. No Drive mount or upload.
BASE = LOCAL_BASE
RAW = BASE / 'data' / 'raw'
OUT = BASE / 'checkpoints'
CORPUS = str(BASE / 'data' / 'input.txt')
CHECKPOINT = str(OUT / 'firstgpt.pt')
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# Generate a deterministic, varied corpus for pipeline verification.
import random
rng = random.Random(20261113)
subjects = ["the small machine", "a curious learner", "the blue engine", "the quiet forest", "a research team", "the little robot", "an observant scientist", "the river system"]
verbs = ["studies", "builds", "measures", "describes", "tests", "discovers", "compares", "records"]
objects = ["patterns in data", "the movement of light", "how memory works", "a sequence of signals", "the structure of language", "changes in temperature", "the flow of information", "a new experiment"]
connectors = ["because", "while", "although", "when", "after", "before", "therefore", "for example"]
details = ["carefully and repeatedly", "using simple observations", "with a clear procedure", "over several experiments", "by checking each result", "without assuming the answer", "and stores the result", "then explains the outcome"]
paragraphs = []
for i in range(1800):
    subject, verb, obj = rng.choice(subjects), rng.choice(verbs), rng.choice(objects)
    connector, detail = rng.choice(connectors), rng.choice(details)
    paragraphs.append(
        f"Example {i + 1}. {subject.capitalize()} {verb} {obj} {connector} "
        f"the process can be examined {detail}. The observation number is {i + 1}, "
        f"and the measured value is {rng.randint(1, 1000)}. "
        "A useful model learns relationships between tokens from many examples."
    )
synthetic_path = RAW / 'synthetic_test_corpus.txt'
synthetic_path.write_text("\n\n".join(paragraphs), encoding='utf-8')
print('Synthetic corpus:', synthetic_path)
print('Paragraphs:', len(paragraphs), '| Characters:', sum(map(len, paragraphs)))
print('Workspace:', BASE, '| Checkpoints:', OUT)

### Prepare the generated corpus
The text file is generated locally in Colab; no file picker is needed.

In [ ]:
if not any(p.is_file() and p.suffix.lower() == '.txt' for p in RAW.rglob('*')):
    raise RuntimeError(f'No .txt corpus files found under {RAW}. Run the corpus-generation cell first, or place a UTF-8 .txt file in the raw folder.')
(BASE / 'data').mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, 'prepare_corpus.py',
    '--input-dir', str(RAW), '--output', CORPUS
], check=True)
if not Path(CORPUS).is_file():
    raise RuntimeError(f'Corpus preparation did not create {CORPUS}')
print('Corpus ready:', CORPUS, '| bytes:', Path(CORPUS).stat().st_size)

### Train from random initialization
Starter configuration for a T4. This is a small baseline, not a claim that the model will become a capable assistant. Training can be resumed using the next cell.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Real T4 training is blocked because CUDA is unavailable. Fix the runtime before training.')
if not Path(CORPUS).is_file():
    raise RuntimeError(f'Corpus file not found: {CORPUS}. Run the generated-corpus preparation cell first.')
subprocess.run([
    sys.executable, 'train.py', '--data', CORPUS, '--out', CHECKPOINT,
    '--steps', '1000', '--eval-interval', '100', '--checkpoint-interval', '250',
    '--batch-size', '16', '--block-size', '256', '--layers', '4', '--heads', '4',
    '--width', '128', '--grad-accum-steps', '2', '--amp'
], check=True)

### Resume training if needed
Run this cell only after a previous training run has created the checkpoint. It uses the same corpus and restores the saved model, optimizer, mixed-precision scaler and RNG states.

In [ ]:
if not Path(CHECKPOINT).is_file():
    raise RuntimeError(f'No resumable checkpoint found at {CHECKPOINT}. Run the initial training cell first.')
subprocess.run([
    sys.executable, 'train.py', '--data', CORPUS, '--out', CHECKPOINT,
    '--resume', '--steps', '1000', '--eval-interval', '100',
    '--checkpoint-interval', '250', '--batch-size', '16',
    '--grad-accum-steps', '2', '--amp'
], check=True)

### Evaluate and generate
Evaluation checks the exact corpus fingerprint. Generation uses the best validation checkpoint.

In [ ]:
BEST_CHECKPOINT = str(OUT / 'firstgpt.best.pt')
if not Path(BEST_CHECKPOINT).is_file():
    raise RuntimeError(f'Best checkpoint not found: {BEST_CHECKPOINT}. Training must finish successfully first.')
subprocess.run([
    sys.executable, 'evaluate.py', '--checkpoint', BEST_CHECKPOINT,
    '--data', CORPUS, '--batches', '50', '--batch-size', '16'
], check=True)
subprocess.run([
    sys.executable, 'generate.py', '--checkpoint', BEST_CHECKPOINT,
    '--prompt', 'The ', '--tokens', '200', '--temperature', '0.8', '--top-k', '40'
], check=True)

### Download a copy of the checkpoints
Download both files to your phone before disconnecting. Temporary Colab storage is erased when the runtime disconnects.

In [ ]:
from google.colab import files
if not Path(CHECKPOINT).is_file() or not Path(BEST_CHECKPOINT).is_file():
    raise RuntimeError('Both latest and best checkpoints must exist before downloading.')
files.download(CHECKPOINT)
files.download(BEST_CHECKPOINT)

## Optional — train Orbit on real open text (no uploads)
The earlier synthetic corpus only tests the training pipeline. This optional stage downloads WikiText-2 and Databricks Dolly 15K directly into temporary Colab storage, then trains FirstGPT from random initialization and instruction-tunes it. No Drive mount or manual upload is involved. Dataset licenses and attribution are recorded in data/SOURCES.md. This is still a small experimental model, not a production-grade assistant.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets>=3.0'], check=True)
REAL_CORPUS = str(BASE / 'data' / 'wikitext_train.txt')
REAL_INSTRUCT_DATA = str(BASE / 'data' / 'orbit_instructions_full.jsonl')
subprocess.run([
    sys.executable, 'download_training_data.py',
    '--wikitext-out', REAL_CORPUS,
    '--wikitext-config', 'wikitext-2-raw-v1',
    '--max-chars', '1000000',
    '--instructions-out', REAL_INSTRUCT_DATA,
    '--max-instructions', '15000',
    '--extra-instructions', str(REPO / 'data' / 'orbit_instructions.example.jsonl')
], check=True)
REAL_CHECKPOINT = str(OUT / 'firstgpt-wikitext.pt')
subprocess.run([
    sys.executable, 'train.py', '--data', REAL_CORPUS, '--out', REAL_CHECKPOINT,
    '--steps', '2000', '--eval-interval', '200', '--checkpoint-interval', '500',
    '--batch-size', '8', '--grad-accum-steps', '2', '--block-size', '256',
    '--layers', '6', '--heads', '8', '--width', '256', '--vocab-size', '512', '--amp'
], check=True)
BEST_CHECKPOINT = str(OUT / 'firstgpt-wikitext.best.pt')
subprocess.run([
    sys.executable, 'evaluate.py', '--checkpoint', BEST_CHECKPOINT,
    '--data', REAL_CORPUS, '--batches', '50', '--batch-size', '8'
], check=True)
print('Real-text base checkpoint:', BEST_CHECKPOINT)


## Optional — Orbit instruction-tuning pipeline
If the real-data stage above ran, this fine-tunes on the downloaded Dolly instruction corpus plus Orbit-specific examples. If you skip that stage, it falls back to the small committed starter set, which is only a smoke test and will not teach useful assistant behaviour.

In [ ]:
INSTRUCT_DATA = Path(REAL_INSTRUCT_DATA) if 'REAL_INSTRUCT_DATA' in globals() and Path(REAL_INSTRUCT_DATA).is_file() else REPO / 'data' / 'orbit_instructions.example.jsonl'
INSTRUCT_CHECKPOINT = str(OUT / 'orbit-instruct.pt')
if not Path(BEST_CHECKPOINT).is_file():
    raise RuntimeError('Train and evaluate FirstGPT first.')
subprocess.run([
    sys.executable, 'finetune_instructions.py',
    '--checkpoint', BEST_CHECKPOINT,
    '--data', str(INSTRUCT_DATA),
    '--out', INSTRUCT_CHECKPOINT,
    '--steps', '1000', '--batch-size', '8', '--eval-interval', '100'
], check=True)
if not Path(INSTRUCT_CHECKPOINT).is_file():
    raise RuntimeError('Instruction tuning did not produce a checkpoint.')
print('Orbit instruction checkpoint:', INSTRUCT_CHECKPOINT)
from google.colab import files
files.download(INSTRUCT_CHECKPOINT)

### Optional — enable extended document, audio and video processing
Installs the optional PDFium and faster-whisper dependencies. Video frame analysis also uses ffmpeg, which is normally present in Colab. Skip this cell if you only need image reading and text documents.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-media.txt")], check=True)
print("Optional PDF OCR and audio-transcription packages installed.")

### Optional — enable image understanding on the Colab T4
This downloads and runs a separate vision model (LLaVA 7B) through Ollama in the same Colab runtime. Orbit can then inspect uploaded images and pass the visual analysis to FirstGPT. This is not native vision training for FirstGPT; it is a connected vision model. The download is several gigabytes and uses temporary Colab storage.

In [ ]:
import os, shutil, subprocess, time, urllib.request
if not shutil.which("ollama"):
    subprocess.run(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"], check=True)
OLLAMA_LOG = open(BASE / "ollama.log", "w")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=OLLAMA_LOG, stderr=subprocess.STDOUT)
OLLAMA_READY = False
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=3) as response:
            if response.status == 200:
                OLLAMA_READY = True
                break
    except Exception:
        time.sleep(2)
if not OLLAMA_READY:
    raise RuntimeError("Ollama did not start. Inspect /content/FirstAI_workspace/ollama.log.")
subprocess.run(["ollama", "pull", "llava:7b"], check=True)
os.environ["ORBIT_VISION_PROVIDER"] = "ollama"
os.environ["ORBIT_VISION_URL"] = "http://127.0.0.1:11434/api/generate"
os.environ["ORBIT_VISION_MODEL"] = "llava:7b"
print("Orbit image understanding is ready through the Colab T4 vision backend.")

### Optional — enable public website browsing
This installs the optional Playwright browser backend. Orbit can open public HTTP(S) sites, read visible text, fill fields without submitting, and click a selector you explicitly specify. Private/local network targets are blocked. This is not a logged-in account automation or unrestricted browser agent.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-browser.txt')], check=True)
subprocess.run([sys.executable, '-m', 'playwright', 'install', 'chromium'], check=True)
print('Orbit public-site browser tools installed. The browser starts on first explicit website command.')

### Optional — enable direct image generation on the T4
Installs the Diffusers backend and configures a public SD 1.5-based DreamShaper checkpoint. The weights download automatically from Hugging Face on the first image request; no ComfyUI server or manual model upload is needed. Image rendering temporarily moves FirstGPT to CPU, uses Diffusers CPU offload, then restores FirstGPT to CUDA. This shares the T4 with optional LLaVA vision, so first render may take a few minutes.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-image.txt')], check=True)
os.environ['ORBIT_IMAGE_BACKEND'] = 'diffusers'
os.environ['ORBIT_IMAGE_MODEL'] = 'Lykon/dreamshaper-8'
os.environ['ORBIT_IMAGE_DEVICE'] = 'cuda'
os.environ['ORBIT_IMAGE_WIDTH'] = '512'
os.environ['ORBIT_IMAGE_HEIGHT'] = '512'
os.environ['ORBIT_IMAGE_STEPS'] = '20'
print('Direct image generation configured. The model will download automatically on the first request.')

## Optional — launch the Orbit interface
This launches the local Orbit prototype using the best checkpoint produced by the stages you ran. If you skipped real-text pretraining and instruction tuning, the interface will use the synthetic Test 3 checkpoint, which is not a capable conversational model. Keep the Colab runtime connected while using the interface.

In [ ]:
from google.colab.output import eval_js
import subprocess, time
os.environ["ORBIT_GPU_MEMORY_FRACTION"] = "0.60" if (os.environ.get("ORBIT_VISION_MODEL") or os.environ.get("ORBIT_IMAGE_BACKEND") == "diffusers") else "0.86"
os.environ["ORBIT_CPU_THREADS"] = "4"
os.environ["ORBIT_AGENT_WORKERS"] = "3"
print("CUDA allocator ceiling:", os.environ["ORBIT_GPU_MEMORY_FRACTION"], "lowered when the separate vision or image-generation model is enabled")
ORBIT_STATE = str(BASE / "orbit_state.sqlite3")
ORBIT_CHECKPOINT = str(OUT / "orbit-instruct.pt")
if not Path(ORBIT_CHECKPOINT).is_file():
    ORBIT_CHECKPOINT = BEST_CHECKPOINT
ORBIT_LOG = open(BASE / "orbit_server.log", "w")
orbit_process = subprocess.Popen([
    sys.executable, "-m", "orbit.server",
    "--checkpoint", ORBIT_CHECKPOINT, "--state", ORBIT_STATE,
    "--host", "0.0.0.0", "--port", "8767"
], cwd=str(REPO), stdout=ORBIT_LOG, stderr=subprocess.STDOUT)
time.sleep(2)
if orbit_process.poll() is not None:
    print((BASE / "orbit_server.log").read_text())
    raise RuntimeError("Orbit server did not start; see the log above.")
ORBIT_URL = eval_js("google.colab.kernel.proxyPort(8767)")
print("Orbit interface:", ORBIT_URL)
print("Keep this Colab runtime connected while using Orbit.")